# LAB 2 — Profiling GPU con Nsight Systems (NSYS)

**Obiettivo:** usare `nsys` per capire *dove va il tempo* in un programma GPU:
- tempo kernel (GPU)
- tempo memcpy (HtoD / DtoH)
- tempo di attesa lato CPU (`cuCtxSynchronize`)

Per leggere i risultati usiamo un parser ah-hoc “user-friendly” (`tools/parse_nsys.py`) invece dell’output completo di `nsys stats`.


## Struttura repo

```
lab2/
  scripts/
    vecadd.py
    stencil9_profile.py
    stencil49_profile.py
    sync_antipattern.py
  tools/
    parse_nsys.py
  results/
    nsys/
  LAB2.ipynb
```


## 0) Setup e requisiti

Serve:
- Linux + GPU NVIDIA + driver
- Python env con `numba`, `numpy`
- Nsight Systems (`nsys`) disponibile sul server

L'ambiente dovrebbe essere gia' installato dopo il lab 1. 
Se serve, vedi le istruzioni di setup nel notebook `lab1`.


In [1]:
# controlliamo che la GPU sia disponibile
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:57:39_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


In [2]:
# Wrap your Numba code in a function or a script
def run_my_numba_code():
    from numba import cuda
    import numpy as np
    
    @cuda.jit
    def add_kernel(a, b, c):
        i = cuda.grid(1)
        if i < a.size:
            c[i] = a[i] + b[i]

    N = 10**6
    a = cuda.to_device(np.ones(N, dtype=np.float32))
    b = cuda.to_device(np.ones(N, dtype=np.float32))
    c = cuda.device_array_like(a)

    add_kernel[3907, 256](a, b, c)
    cuda.synchronize()

# Save the code to a temporary file
with open('profile_me.py', 'w') as f:
    f.write("""
import numpy as np
from numba import cuda
@cuda.jit
def add_kernel(a, b, c):
    i = cuda.grid(1)
    if i < a.size:
        c[i] = a[i] + b[i]

N = 10**6
a = cuda.to_device(np.ones(N, dtype=np.float32))
b = cuda.to_device(np.ones(N, dtype=np.float32))
c = cuda.device_array_like(a)
add_kernel[3907, 256](a, b, c)
cuda.synchronize()
    """)

# Run the profile command
!nsys profile -o my_numba_report --force-overwrite=true python profile_me.py

Try the 'nsys status --environment' command to learn more.

Try the 'nsys status --environment' command to learn more.

Generating '/tmp/nsys-report-1b45.qdstrm'
[1/1] [========================100%] my_numba_report.nsys-rep
Generated:
	/home/e4user/Documents/s362027/lab2/my_numba_report.nsys-rep


In [3]:
# Verifiche base
!nvidia-smi
!python -c "import numpy as np; import numba; from numba import cuda; print('Numba:', numba.__version__); print('GPU:', cuda.get_current_device().name)"
!nsys --version

Thu Mar  5 11:17:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.95.05              Driver Version: 580.95.05      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GB10                    On  |   0000000F:01:00.0  On |                  N/A |
| N/A   45C    P0             13W /  N/A  | Not Supported          |      4%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1) Comandi da usare

### Profilare da linea di comando (genera `.nsys-rep`)
```bash
!mkdir -p results/nsys
!rm -f results/nsys/exp.nsys-rep
!nsys profile --force-overwrite true -t cuda,osrt -o results/nsys/exp python scripts/EXP.py
```

### Riassunto “pulito”, basato sul parser custom in Python (`tools/parse_nsys.py`)
```bash
python tools/parse_nsys.py results/nsys/exp.nsys-rep
```
l’output di `nsys stats` è lungo: lo useremo solo come approfondimento.


## 2) TODO 1 — Scrivere un kernel semplicissimo: Vector Add

Script: `scripts/vecadd.py`

**Cosa fare:**
- Completare il kernel GPU per sommare 2 vettori (`C = A + B`)
- Eseguire e profilare


In [4]:
!mkdir -p results/nsys
!rm -f results/nsys/exp.nsys-rep

In [5]:
# Run (no profiling)
!python ./scripts/vec_add.py

vec_add correct=True | n=10000000 | grid=39063 block=256


In [6]:
# Profiling + summary
!rm -f ./results/nsys/vecadd.nsys-rep
!nsys profile --force-overwrite true -t cuda,osrt -o ./results/nsys/vecadd python ./scripts/vec_add.py
!python ./tools/parse_nsys.py ./results/nsys/vecadd.nsys-rep

Try the 'nsys status --environment' command to learn more.

Try the 'nsys status --environment' command to learn more.

vec_add correct=True | n=10000000 | grid=39063 block=256
Generating '/tmp/nsys-report-d823.qdstrm'
[1/1] [========================100%] vecadd.nsys-rep
Generated:
	/home/e4user/Documents/s362027/lab2/./results/nsys/vecadd.nsys-rep
=== NSYS SUMMARY (student-friendly) ===
CPU wait (cuCtxSynchronize): 1.302 ms
GPU kernel total           : 1.021 ms
GPU memcpy total           : 2.109 ms (HtoD 1.332 ms, DtoH 0.777 ms)

Top 1 kernels by total GPU time:
  1. vec_add
     total: 1.021 ms | instances: 2
     avg per launch: 0.510 ms


`nsys stats`: output completo (non useremo ora)


In [7]:
!nsys stats --force-export=true ./results/nsys/vecadd.nsys-rep

Generating SQLite file ./results/nsys/vecadd.sqlite from ./results/nsys/vecadd.nsys-rep
Processing [./results/nsys/vecadd.sqlite] with [/opt/nvidia/nsight-systems/2025.3.2/host-linux-armv8/reports/nvtx_sum.py]... 
SKIPPED: ./results/nsys/vecadd.sqlite does not contain NV Tools Extension (NVTX) data.

Processing [./results/nsys/vecadd.sqlite] with [/opt/nvidia/nsight-systems/2025.3.2/host-linux-armv8/reports/osrt_sum.py]... 

 ** OS Runtime Summary (osrt_sum):

 Time (%)  Total Time (ns)  Num Calls   Avg (ns)    Med (ns)   Min (ns)   Max (ns)    StdDev (ns)            Name         
 --------  ---------------  ---------  -----------  ---------  --------  -----------  ------------  ----------------------
     69.6      471,195,136         95  4,959,948.8  746,608.0     3,376  133,806,192  17,110,409.3  poll                  
     26.9      181,856,032        438    415,196.4    9,552.0     1,024    7,628,080   1,043,707.5  ioctl                 
      1.5       10,257,696         18    56

## 3) TO-DO 2 — Matrix Multiplication CUDA: naive vs tiled (shared memory)

L'obiettivo e' implementere due versioni della moltiplicazione di matrici quadrate `C = A × B` e profilare le implementazioni GPU con **Nsight Systems**. Le versioni da implementare sono:

1. una versione **CPU** di riferimento ( `matmul_cpu(A, B)` )
2. una versione **CUDA naive** (solo global memory)
3. una versione **CUDA tiled** con **shared memory**

E' richiesto di completare il file `matmul_profile.py` con le implementazioni mancanti, verificare la correttezza dei risultati e profilare con `nsys` per confrontare i tempi di esecuzione.

### Parte 1 — Implementazione CPU (TODO)

Scrivere una funzione CPU che calcoli:

```math
C[i,j] = \sum_k A[i,k] \cdot B[k,j]
```

Requisiti:
- input: `A, B` NumPy `float32` di dimensione `n×n`
- output: `C` NumPy `float32`
- usare **due o tre cicli annidati**
- NON usare `A @ B` o funzioni BLAS

---

### Parte 2 — Kernel CUDA naive (TODO)

Scrivere un kernel `@cuda.jit matmul_naive(...)` CUDA che:
- usa una **griglia 2D** (`cuda.grid(2)`)
- ogni thread calcola **un elemento** `C[i,j]`
- legge `A` e `B` **solo dalla memoria globale**
- non usa shared memory

---

### Parte 3 — Kernel CUDA tiled con shared memory (TODO)

Scrivere un kernel CUDA che:
- divide le matrici in **tile quadrati** con tile size `TILE × TILE` ( `block = (TILE, TILE)`)
- carica porzioni di `A` e `B` in **shared memory**
- sincronizza i thread del blocco (`cuda.syncthreads()`)
- accumula il risultato parziale

---

### Parte 4 — Verifica di correttezza

Per ogni implementazione:
- confrontare il risultato GPU con la versione CPU
- usare `np.allclose` con tolleranze adeguate per `float32`
- riportare il massimo errore assoluto osservato

---

### Parte 5 — Benchmark e profiling

Profilare il programma con **Nsight Systems** e misurare:
- tempo CPU (una sola esecuzione)
- tempo GPU naive (media su più iterazioni)
- tempo GPU tiled (media su più iterazioni)


```bash
nsys profile -t cuda,nvtx,osrt -o results/nsys/matmul python scripts/matmul_profile.py


In [19]:
!rm -f ./results/nsys/matmul_profile.py
!nsys profile --force-overwrite true -t cuda,osrt -o ./results/nsys/matmul_profile python ./scripts/matmul_profile.py
!python ./tools/parse_nsys.py ./results/nsys/matmul_profile.nsys-rep

Try the 'nsys status --environment' command to learn more.

Try the 'nsys status --environment' command to learn more.

Numba CUDA MatMul profiling script (+ optional CuPy/cuBLAS)
GPU: NVIDIA GB10
TILE = 16
CuPy available: True

=== n = 512 ===
CPU (NumPy BLAS)  : 4.99 ms (single run)
GPU naive         : 1.56 ms/iter | correct=True | max_abs_err=0.0001068
GPU tiled(shared) : 1.51 ms/iter | correct=True | max_abs_err=0.0001068
Speedup tiled vs naive: 1.03x
GPU register-blocked  : 1.48 ms/iter | correct=True | max_abs_err=0.0001068
Speedup register-blocked vs naive: 1.05x
Speedup register-blocked vs tiled: 1.02x
GPU double-buffered   : 1.48 ms/iter | correct=True | max_abs_err=0.0001068
Speedup double-buffered vs tiled: 1.02x
GPU CuPy (cuBLAS) : 0.05 ms/iter | correct=True | max_abs_err=0.0001144
Speedup cuBLAS vs tiled: 30.45x

=== n = 1024 ===
CPU (NumPy BLAS)  : 3.95 ms (single run)
GPU naive         : 11.75 ms/iter | correct=True | max_abs_err=0.0002136
GPU tiled(shared) : 11.61 ms/i

In [9]:
!nsys stats --force-export=true ./results/nsys/matmul_profile.nsys-rep

Generating SQLite file ./results/nsys/matmul_profile.sqlite from ./results/nsys/matmul_profile.nsys-rep
Processing [./results/nsys/matmul_profile.sqlite] with [/opt/nvidia/nsight-systems/2025.3.2/host-linux-armv8/reports/nvtx_sum.py]... 
SKIPPED: ./results/nsys/matmul_profile.sqlite does not contain NV Tools Extension (NVTX) data.

Processing [./results/nsys/matmul_profile.sqlite] with [/opt/nvidia/nsight-systems/2025.3.2/host-linux-armv8/reports/osrt_sum.py]... 

 ** OS Runtime Summary (osrt_sum):

 Time (%)  Total Time (ns)  Num Calls     Avg (ns)         Med (ns)        Min (ns)        Max (ns)       StdDev (ns)             Name         
 --------  ---------------  ---------  ---------------  ---------------  -------------  --------------  ---------------  ----------------------
     46.4  303,820,954,016         77  3,945,726,675.5    754,279,216.0        116,768  13,209,790,976  5,348,385,087.9  pthread_cond_wait     
     18.0  118,032,336,816     19,975      5,909,003.1      4,5

## 4) TO-DO 3: — Riduzione: `sum/mean` (CuPy vs Numba)

In questo esercizio confrontiamo una riduzione ottimizzata (CuPy) con una riduzione “naive” scritta con Numba CUDA. Le riduzioni sono difficili da ottimizzare e perché una libreria (CuPy) può essere molto più veloce di un kernel scritto “a mano” senza ottimizzazioni avanzate. Il codice si trova in `scripts/reduction_profile.py`.

---

### Task A — CuPy

Dato un vettore 1D `x` (float32, molto grande, es. `N = 50_000_000`):

1. Calcolare `s_cupy = cp.sum(x) \ cp.mean(x)`.
2. Misurare il tempo medio su almeno 30 iterazioni (con warmup).
3. Sincronizzare correttamente la GPU prima di fermare il timer.

---

### Task B — Numba naive (partial sums per blocco)

Implementare una riduzione in due fasi:

1. **Kernel CUDA** `block_sum_kernel(x, partial)`:

   * ogni blocco accumula una somma parziale in `partial[blockIdx.x]`
   * usare `cuda.grid(1)` e un loop a stride per coprire tutto `x`

2. **Finalizzazione su CPU**:

   * copiare `partial` su host
   * sommare su CPU per ottenere `s_numba`

---

### Verifiche

* Confrontare entrambe le somme con un riferimento CPU accurato: `np.sum(x_host, dtype=np.float64) \ np.mean(x_host, dtype=np.float64)`.
* Calcolare errore relativo:
   ```math
  \frac{|s - s_{ref}|}{|s_{ref}|}
   ```




In [10]:
!rm -f ./results/nsys/reduction_profile.py
!nsys profile --force-overwrite true -t cuda,osrt -o ./results/nsys/reduction_profile python ./scripts/reduction_profile.py
!python ./tools/parse_nsys.py ./results/nsys/reduction_profile.nsys-rep

Try the 'nsys status --environment' command to learn more.

Try the 'nsys status --environment' command to learn more.

/home/e4user/Documents/s362027/lab2/cuda_lab/lib/python3.12/site-packages/numba/cuda/dispatcher.py:536: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))
/home/e4user/Documents/s362027/lab2/cuda_lab/lib/python3.12/site-packages/numba/cuda/dispatcher.py:536: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))
/home/e4user/Documents/s362027/lab2/cuda_lab/lib/python3.12/site-packages/numba/cuda/dispatcher.py:536: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(NumbaPerformanceWarning(msg))
=== Reduction: r = sum(x) / mean(x) ===
N = 50000000
CPU ref (float64)       : r=50000000.000000

CuPy (sum+mean)         : 1.862 ms/iter | r=500000

## 5) Stencil kernel

Uno stencil è un’operazione che calcola ogni elemento di una matrice usando i **valori vicini**.

Gli stencil sono molto comuni in:

* image processing
* computer vision
* simulazioni fisiche
* AI (pre-processing, convoluzioni)

---

## Stencil 3×3 (9-point)

Per ogni elemento `(y, x)` della matrice di input:

* legge i **9 valori** in un intorno 3×3
* calcola la **media**
* scrive il risultato nella matrice di output

Visivamente:

```
x x x
x O x
x x x
```

Dove `O` è il punto calcolato e `x` sono i vicini.

### Perché lo usiamo

* è **semplice**
* ha **poco riuso dei dati**
* spesso la cache GPU è già sufficiente

Se il riuso è basso, l’overhead può peggiorare le prestazioni.

Nel laboratorio:

* la versione “shared” può essere **uguale o più lenta** della naive

---

## Stencil 7×7 (49-point)

### Cosa fa

Per ogni elemento `(y, x)`:

* legge i **49 valori** in un intorno 7×7
* calcola la **media**
* scrive il risultato

Visivamente:

```
x x x x x x x
x x x x x x x
x x x x x x x
x x x O x x x
x x x x x x x
x x x x x x x
x x x x x x x
```

### Perché è diverso dal 3×3

* ogni output legge **molti più dati**
* il costo delle letture di memoria è molto più alto
* il **riuso dei dati** tra thread aumenta

> Quando il riuso dei dati è alto, caricare un blocco in shared memory può diventare conveniente.

Nel laboratorio:

* la versione “shared” può mostrare un **miglioramento misurabile** (ma non sempre...)
## Versione naive vs versione shared

### Versione naive

* ogni thread legge tutti i dati direttamente dalla **global memory**
* codice semplice
* zero sincronizzazioni
* nessuna cooperazione tra thread

### Versione shared

* i thread di un blocco **collaborano**
* caricano una porzione della matrice in **shared memory**
* riusano i dati più volte
* richiede:

  * logica aggiuntiva
  * sincronizzazione (`cuda.syncthreads()`)

La shared memory **non è sempre migliore**, ma conviene quando il riuso dei dati compensa l’overhead.


### Parte 1 — Shared memory NON sempre aiuta (3×3 / 9-point)

Script: `scripts/stencil9_profile.py`

**Obiettivo:** vedere un caso in cui la versione “shared” può essere uguale o peggiore della naive (overhead > beneficio).

Esegui lo script e annota `GPU naive` e `GPU shared`:



In [11]:
!python scripts/stencil9_profile.py

Numba CUDA 9-point stencil (3x3 blur) profiling script
GPU: NVIDIA GB10
Block naive  : (32, 8)
Block shared : (32, 8)  shared tile = (10, 34) with halo=1

=== size = 2048 x 2048 ===
CPU ref (NumPy)      : 11.58 ms (single run)
GPU naive            : 0.4748 ms/iter | correct=True | max_abs_err=1.79e-07
GPU shared(tiling)   : 0.5117 ms/iter | correct=True | max_abs_err=1.79e-07
Speedup shared vs naive: 0.93x

=== size = 4096 x 4096 ===
CPU ref (NumPy)      : 48.54 ms (single run)
GPU naive            : 1.8380 ms/iter | correct=True | max_abs_err=1.79e-07
GPU shared(tiling)   : 2.0162 ms/iter | correct=True | max_abs_err=1.79e-07
Speedup shared vs naive: 0.91x



In [12]:
!rm -f results/nsys/stencil9.nsys-rep
!nsys profile --force-overwrite true -t cuda,osrt -o results/nsys/stencil9 python scripts/stencil9_profile.py
!python tools/parse_nsys.py results/nsys/stencil9.nsys-rep

Try the 'nsys status --environment' command to learn more.

Try the 'nsys status --environment' command to learn more.

Numba CUDA 9-point stencil (3x3 blur) profiling script
GPU: NVIDIA GB10
Block naive  : (32, 8)
Block shared : (32, 8)  shared tile = (10, 34) with halo=1

=== size = 2048 x 2048 ===
CPU ref (NumPy)      : 15.72 ms (single run)
GPU naive            : 0.4597 ms/iter | correct=True | max_abs_err=1.79e-07
GPU shared(tiling)   : 0.5191 ms/iter | correct=True | max_abs_err=1.79e-07
Speedup shared vs naive: 0.89x

=== size = 4096 x 4096 ===
CPU ref (NumPy)      : 67.66 ms (single run)
GPU naive            : 1.8399 ms/iter | correct=True | max_abs_err=1.79e-07
GPU shared(tiling)   : 2.0261 ms/iter | correct=True | max_abs_err=1.79e-07
Speedup shared vs naive: 0.91x

Generating '/tmp/nsys-report-56fb.qdstrm'
[1/1] [========================100%] stencil9.nsys-rep
Generated:
	/home/e4user/Documents/s362027/lab2/results/nsys/stencil9.nsys-rep
=== NSYS SUMMARY (student-friendly) =

**Domande:**
- Quale kernel è più veloce?

    stencil9_naive è più veloce: 1.095 ms/launch vs 1.198 ms/launch della versione shared (~9% più lento con shared memory). Lo speedup misurato è 0.89–0.92x. La shared memory peggiora le prestazioni perché lo stencil 3×3 ha un riuso dei dati troppo basso (ogni thread legge solo 9 elementi) — il costo di caricare il tile in shared memory e sincronizzare i thread supera il beneficio del riuso.

- Le memcpy contano o no?

    Contano, ma non dominano: 130 ms su ~1009 ms totali di kernel (~13%). Quasi tutto il trasferimento è DtoH (128.7 ms), probabilmente dovuto alle copie di verifica correttezza su ogni dimensione. HtoD è trascurabile (1.4 ms). Se si rimuovesse la verifica, le memcpy diventerebbero irrilevanti.
    
- `cuCtxSynchronize` domina? perché?

    Sì: 1003 ms vs 1009 ms kernel total — quasi tutto il wall-time CPU è attesa GPU. Questo è normale e atteso: cuCtxSynchronize blocca la CPU finché la GPU non ha terminato tutto il lavoro accodato. Il tempo di attesa è circa uguale al tempo kernel totale perché la CPU non fa nulla di utile nel frattempo — lancia i kernel e poi aspetta. Non è un anti-pattern qui (la sync è fatta una sola volta per misurare), ma mostra che il programma è completamente compute-bound lato GPU.

### Parte 2 — Shared memory può aiutare (7×7 / 49-point)

Script: `scripts/stencil49_profile.py`

**Obiettivo:** vedere un caso in cui la versione “shared” ha senso (riuso alto).

In [13]:
# Run (no profiling)
!python scripts/stencil49_profile.py --mode both

Numba CUDA 49-point stencil (7x7 blur)
GPU: NVIDIA GB10
Block naive  : (32, 8)
Block shared : (32, 8)  shared tile = (14, 38)

=== size = 2048 x 2048 ===
GPU naive  : 2.261 ms | correct=True
GPU shared : 2.294 ms | correct=True
Speedup shared vs naive: 0.99x

=== size = 4096 x 4096 ===
GPU naive  : 9.103 ms | correct=True
GPU shared : 9.142 ms | correct=True
Speedup shared vs naive: 1.00x



In [14]:
# Profile naive
!rm -f results/nsys/stencil49_naive.nsys-rep
!nsys profile --force-overwrite true -t cuda,osrt -o results/nsys/stencil49_naive python scripts/stencil49_profile.py
!python tools/parse_nsys.py results/nsys/stencil49_naive.nsys-rep

Try the 'nsys status --environment' command to learn more.

Try the 'nsys status --environment' command to learn more.

Numba CUDA 49-point stencil (7x7 blur)
GPU: NVIDIA GB10
Block naive  : (32, 8)
Block shared : (32, 8)  shared tile = (14, 38)

=== size = 2048 x 2048 ===
GPU naive  : 2.285 ms | correct=True
GPU shared : 2.312 ms | correct=True
Speedup shared vs naive: 0.99x

=== size = 4096 x 4096 ===
GPU naive  : 9.122 ms | correct=True
GPU shared : 9.180 ms | correct=True
Speedup shared vs naive: 0.99x

Generating '/tmp/nsys-report-1be0.qdstrm'
[1/1] [========================100%] stencil49_naive.nsys-rep
Generated:
	/home/e4user/Documents/s362027/lab2/results/nsys/stencil49_naive.nsys-rep
=== NSYS SUMMARY (student-friendly) ===
CPU wait (cuCtxSynchronize): 2514.497 ms
GPU kernel total           : 2514.503 ms
GPU memcpy total           : 128.522 ms (HtoD 1.412 ms, DtoH 127.110 ms)

Top 2 kernels by total GPU time:
  1. stencil49_shared
     total: 1261.832 ms | instances: 220
     

**Domande:**
- Quale kernel è più veloce?

    stencil49_shared è marginalmente più veloce: 5.498 ms vs 5.514 ms/launch, speedup 1.00x — praticamente identici. Rispetto allo stencil 3×3, la shared memory non penalizza più (il riuso dei dati è maggiore: 49 elementi per output invece di 9), ma non porta ancora un vantaggio misurabile. Il beneficio del riuso compensa l'overhead, ma non lo supera significativamente.

- Le memcpy contano o no?

    No, sono trascurabili: 119 ms su ~2422 ms totali (~5%). HtoD è 1.4 ms (irrilevante), DtoH 117.8 ms dovuto alle copie di verifica per correttezza. Rimuovendo le verifiche, le memcpy scomparirebbero quasi del tutto.

- `cuCtxSynchronize` domina? perché?

    Sì: 2421 ms di attesa CPU vs 2422 ms di kernel GPU — quasi tutto il wall-time è sincronizzazione. Il motivo è lo stesso dello stencil9: la CPU lancia i kernel e poi attende che la GPU finisca. Il tempo cuCtxSynchronize ≈ tempo kernel totale indica che il programma è compute-bound — la GPU è sempre occupata e la CPU non fa altro che aspettarla. Non è un anti-pattern (non c'è sync dentro al loop), è semplicemente la struttura corretta per misurare il tempo di esecuzione GPU.

## 5) TO-DO 3 - Anti-pattern: sincronizzare nel posto sbagliato

Script: `scripts/sync_antipattern.py`

**Obiettivo:** far vedere che `cuda.synchronize()` dentro al loop serializza tutto e aumenta l’attesa CPU.

Useremo:
- `--mode good`
- `--mode bad`

Completare, profilare e confrontare i summary delle due versioni


### Anti-pattern: sincronizzare nel posto sbagliato (`cuda.synchronize()`)

### Cos’è una sincronizzazione

`cuda.synchronize()` forza la **CPU ad aspettare** finché la GPU non ha terminato **tutto** il lavoro lanciato fino a quel punto.

Normalmente:

* la CPU **lancia** un kernel GPU
* la GPU **lavora in parallelo**
* la CPU può continuare a fare altro

Nella versione “good”:

* i kernel vengono lanciati in un loop
* **una sola sincronizzazione** viene fatta alla fine, per misurare il tempo totale

Pseudo-codice:

```python
for i in range(N):
    launch_kernel()
cuda.synchronize()   # una sola volta
```

la GPU lavora in modo continuo, mentre la CPU puo' continuare a fare altro. La pipeline è efficiente

## Anti-pattern (BAD): sincronizzare dentro al loop

Nella versione “bad”:

* la CPU lancia il kernel
* **subito dopo si ferma ad aspettare**
* questo succede ad ogni iterazione

Pseudo-codice:

```python
for i in range(N):
    launch_kernel()
    cuda.synchronize()   # ANTI-PATTERN
```


la GPU viene usata in modo **seriale**. Si perde completamente il vantaggio dell’asincronia.

---

Nel profiling con Nsight Systems:

* la riga `cuCtxSynchronize` **domina il tempo totale**
* il tempo kernel può essere identico tra GOOD e BAD
* ma il tempo totale peggiora

In applicazioni reali (AI, vision, simulazioni),  sincronizzare troppo spesso è un anti-pattern comune che: riduce throughput, aumenta latenza, spreca GPU costose

In [15]:
# Run (no profiling)
!python scripts/sync_antipattern.py --mode good
!python scripts/sync_antipattern.py --mode bad

mode=good | 0.058 ms/iter | correct=True | n=2000000
mode=bad | 0.244 ms/iter | correct=True | n=2000000


In [16]:
# Profile GOOD
!rm -f results/nsys/sync_good.nsys-rep
!nsys profile --force-overwrite true -t cuda,osrt -o results/nsys/sync_good python scripts/sync_antipattern.py --mode good
!python tools/parse_nsys.py results/nsys/sync_good.nsys-rep

# Profile BAD
!rm -f results/nsys/sync_bad.nsys-rep
!nsys profile --force-overwrite true -t cuda,osrt -o results/nsys/sync_bad python scripts/sync_antipattern.py --mode bad
!python tools/parse_nsys.py results/nsys/sync_bad.nsys-rep

Try the 'nsys status --environment' command to learn more.

Try the 'nsys status --environment' command to learn more.

mode=good | 0.065 ms/iter | correct=True | n=2000000
Generating '/tmp/nsys-report-3d66.qdstrm'
[1/1] [========================100%] sync_good.nsys-rep
Generated:
	/home/e4user/Documents/s362027/lab2/results/nsys/sync_good.nsys-rep
=== NSYS SUMMARY (student-friendly) ===
CPU wait (cuCtxSynchronize): 66.794 ms
GPU kernel total           : 1215.667 ms
GPU memcpy total           : 0.420 ms (HtoD 0.282 ms, DtoH 0.137 ms)

Top 1 kernels by total GPU time:
  1. saxpy
     total: 1215.667 ms | instances: 20020
     avg per launch: 0.061 ms
Try the 'nsys status --environment' command to learn more.

Try the 'nsys status --environment' command to learn more.

mode=bad | 0.217 ms/iter | correct=True | n=2000000
Generating '/tmp/nsys-report-40a4.qdstrm'
[1/1] [========================100%] sync_bad.nsys-rep
Generated:
	/home/e4user/Documents/s362027/lab2/results/nsys/sync_bad.nsy

## 6) Block size sweep (stencil49)

`scripts/stencil49_sweep.py`

Questo script serve a **testare diverse configurazioni di block size** per lo stesso kernel GPU (`stencil49_shared`, blur 7×7).

Il kernel **non cambia**: cambia solo **come viene lanciato** sulla GPU
(numero di thread per blocco in X e Y).

Questo esercizio mostra che **la launch configuration è parte dell’ottimizzazione**.

---

### Cosa fa lo script

`stencil49_sweep.py` fa due cose distinte:

1. **Sweep con timing (veloce)**
   Prova più configurazioni di block size e stampa una **tabella comparativa**:

   * tempo medio per iterazione (ms)
   * speedup rispetto al kernel naive

2. **Profiling pulito con NSYS (opzionale)**
   Permette di profilare **una sola configurazione scelta**, evitando report confusi.

---

### Step A — Sweep veloce (solo timing)

Esegui lo sweep e guarda la tabella finale
(**ms/iter più basso = meglio**):

```bash
python scripts/stencil49_sweep.py --h 2048 --w 2048
```

Per default lo script prova queste configurazioni (BX×BY):

* `32×8`  (baseline)
* `32×16`
* `16×16`

---

### TODO

Aggiungi **1–2 configurazioni tue** (es. `64×4`, `8×32`) passando l’argomento `--configs`:

```bash
python scripts/stencil49_sweep.py --h 2048 --w 2048 --configs "32x8,32x16,16x16,64x4,8x32"
```

Annota:

* quale configurazione è la più veloce
    32   16      1.972 ms/iter              
* di quanto migliora rispetto alla baseline
    1.02 
---

### Step B — Profiling con NSYS (solo la configurazione migliore)

Scegli **una sola** configurazione dallo sweep (esempio: `16×16`)
e profila **solo quella**, per avere un report chiaro:

```bash
rm -f results/nsys/st49_best.nsys-rep
nsys profile --force-overwrite true -t cuda,osrt -o results/nsys/st49_best \
  python scripts/stencil49_sweep.py --h 2048 --w 2048 --block 16,16 --only-shared
python tools/parse_nsys.py results/nsys/st49_best.nsys-rep
```

---

### Cosa osservare nel riassunto NSYS

* **tempo totale kernel GPU**
  275.262 ms
* **tempo di attesa CPU (`cuCtxSynchronize`)**
  274.749 ms
* assenza di memcpy dominanti (in questo caso)
  0.288 ms (HtoD 0.288 ms, DtoH 0.000 ms)
---

### Cosa consegnare

1. tabella dello sweep (copiata dall’output del terminale)
2. block size scelto come “best”
3. output del parser NSYS per la configurazione best

---


# Tabella dello sweep

### System Configuration
*   **GPU:** NVIDIA GB10
*   **Stencil:** 7x7 (radius=3)
*   **Domain Size:** 2048 $\times$ 2048
*   **Execution:** 120 iterations (20 warmup)

---

### Baseline: Naive Implementation
*   **Block Dimensions:** (32, 8)
*   **Time:** 1.941 ms/iter
*   **Status:** Correct

---

### Shared Kernel Block Sweep
*Lower time is better.*

| BX | BY | Time (ms/iter) | Speedup (vs Naive) | Correct |
| :---: | :---: | :---: | :---: | :---: |
| **32** | **16** | **1.972** | **1.02** | **True** |
| 32 | 8 | 1.981 | 1.02 | True |
| 16 | 16 | 1.991 | 1.01 | True |
| 8 | 32 | 2.001 | 1.01 | True |
| 64 | 4 | 2.062 | 0.98 | True |

# Block size scelto come “best”

python3 scripts/stencil49_sweep.py --h 2048 --w 2048 --configs '32x16'

### System Configuration
*   **GPU:** NVIDIA GB10
*   **Stencil:** 7x7 (radius=3)
*   **Domain Size:** 2048 $\times$ 2048
*   **Execution:** 120 iterations (20 warmup)

---

### Baseline: Naive Implementation
*   **Block Dimensions:** (32, 8)
*   **Time:** 1.940 ms/iter
*   **Status:** Correct

---

### Shared Kernel Block Sweep
*Lower time is better.*

| BX | BY | Time (ms/iter) | Speedup (vs Naive) | Correct |
| :---: | :---: | :---: | :---: | :---: |
| 32 | 16 | 1.967 | 1.02 | True |


# Output del parser NSYS per la configurazione best

=== NSYS SUMMARY (student-friendly) ===

CPU wait (cuCtxSynchronize): 274.749 ms

GPU kernel total           : 275.262 ms

GPU memcpy total           : 0.288 ms (HtoD 0.288 ms, DtoH 0.000 ms)

Top 1 kernels by total GPU time:
  1. stencil49_naive
     total: 275.262 ms | instances: 140
     avg per launch: 1.966 ms

# 8. Conclusioni, Domande aperte e Consegna

Il notebook di lavoro del lab deve essere consegnato nella sezione 'elaborati' sul portale della didattica. Questo deve contenere il codice compeleto e i grafici richiesti. Sono richiesti gli output di tempi e correctness per:
- vector add (TODO 1)
- matmul naive vs tiled (TODO 2)
- stencil9
- stencil49 (naive vs shared)
- sync antipattern (good vs bad)

---

### Vector Add

| Componente        | Tempo (ms) | Percentuale |
| ----------------- | ---------- | ----------- |
| Kernel CUDA       | 1.021      | 32.6%       |
| Trasferimenti H→D | 1.332      | 42.6%       |
| Trasferimenti D→H | 0.777      | 24.8%       |
| Altro / CPU       | ~0         | ~0%         |

> **Transfer-bound**: il kernel è quasi istantaneo, i trasferimenti dominano (~67%). Per un vettore grande con un'operazione banale (A+B) il tempo di calcolo è trascurabile rispetto al movimento dei dati.

---

### Matrix Multiplication (naive + tiled)

| Componente        | Tempo (ms) | Percentuale |
| ----------------- | ---------- | ----------- |
| Kernel CUDA       | 58846.353  | 99.25%      |
| Trasferimenti H→D | 6.429      | 0.01%       |
| Trasferimenti D→H | 440.049    | 0.74%       |
| Altro / CPU       | ~0         | ~0%         |

> **Compute-bound**: il calcolo O(n³) su ~140 iterazioni ammortizza completamente il trasferimento O(n²). I trasferimenti sono irrilevanti.

---

### Riduzione (sum/mean)

| Componente        | Tempo (ms) | Percentuale |
| ----------------- | ---------- | ----------- |
| Kernel CUDA       | 120.525    | 94.1%       |
| Trasferimenti H→D | 6.908      | 5.4%        |
| Trasferimenti D→H | 0.649      | 0.5%        |
| Altro / CPU       | ~0         | ~0%         |

> **Compute-bound** con trasferimento H→D non trascurabile (~5%). Il vettore (50M elementi) è grande: la copia iniziale pesa. Le riduzioni sono memory-bandwidth bound lato GPU.

---

### Stencil 3×3 / 9-point

| Componente        | Tempo (ms) | Percentuale |
| ----------------- | ---------- | ----------- |
| Kernel CUDA       | 1060.125   | 87.7%       |
| Trasferimenti H→D | 1.419      | 0.1%        |
| Trasferimenti D→H | 146.634    | 12.1%       |
| Altro / CPU       | ~0         | ~0%         |

> **Compute-bound** (il kernel domina), ma D→H pesa ~12% per le copie di verifica correttezza su ogni dimensione. Il kernel naive è più veloce dello shared (~9-11%), poiché il riuso dei dati nel 3×3 è troppo basso per compensare l'overhead.

---

### Stencil 7×7 / 49-point

| Componente        | Tempo (ms) | Percentuale |
| ----------------- | ---------- | ----------- |
| Kernel CUDA       | 2514.503   | 95.1%       |
| Trasferimenti H→D | 1.412      | 0.05%       |
| Trasferimenti D→H | 127.110    | 4.8%        |
| Altro / CPU       | ~0         | ~0%         |

> **Compute-bound**: il kernel occupa ~95% del tempo. Nonostante il riuso maggiore (49 elementi per output), la versione shared non porta vantaggi apprezzabili rispetto alla naive (~1% di differenza), perché la L2 cache della GPU gestisce già bene i pattern di accesso regolari.

---

### Sync Antipattern — GOOD (sync fuori dal loop)

| Componente        | Tempo (ms) | Percentuale |
| ----------------- | ---------- | ----------- |
| Kernel CUDA       | 1215.667   | 99.97%      |
| Trasferimenti H→D | 0.282      | 0.02%       |
| Trasferimenti D→H | 0.137      | 0.01%       |
| Altro / CPU       | ~0         | ~0%         |

> **Compute-bound**, CPU quasi libera: `cuCtxSynchronize` = 66 ms << kernel = 1215 ms. La GPU lavora in modo continuo e la CPU non viene bloccata ad ogni iterazione.

---

### Sync Antipattern — BAD (sync dentro il loop)

| Componente             | Tempo (ms) | Percentuale |
| ---------------------- | ---------- | ----------- |
| Kernel CUDA            | 497.800    | 18.4%       |
| Trasferimenti H→D      | 0.274      | 0.01%       |
| Trasferimenti D→H      | 0.134      | 0.005%      |
| Altro / CPU (sync OH)  | 2209.434   | 81.6%       |

> **Sync-overhead bound**: la CPU viene bloccata 20020 volte (una per iterazione), causando 2209 ms di overhead aggiuntivo. Il kernel GPU fa lo stesso lavoro della versione GOOD ma il wall time totale è ~3.3× più alto. Questo è l'anti-pattern per eccellenza.

---

Scrivere **5–10 righe** che descrivano:

Il pattern dominante varia per esperimento: la **vecadd** è transfer-bound (calcolo banale, trasferimenti costosi), mentre **matmul**, **stencil9/49** e **reduction** sono compute-bound (il kernel GPU occupa 88–99% del tempo). In tutti i casi compute-bound, `cuCtxSynchronize ≈ GPU kernel total`, confermando che la CPU aspetta la GPU senza fare altro. I trasferimenti D→H pesano visibilmente in stencil (12%) e vecadd (25%) a causa delle copie di verifica per correttezza, ma sarebbero trascurabili in produzione. L'esperimento più istruttivo è il sync antipattern: a parità di lavoro GPU, la versione BAD degrada da compute-bound a **sync-overhead bound** (81% del tempo è overhead CPU), dimostrando come un uso scorretto di `cuda.synchronize()` possa vanificare completamente i benefici del parallelismo GPU.

---

Rispondere alle seguenti domande (brevemente, indicativamente 3-5 righe per domanda):

- **Cosa indica `cuCtxSynchronize`?**
Indica una sincronizzazione esplicita del contesto GPU: la CPU si blocca finché tutte le operazioni GPU accodate non sono completate. Nel profiling NSYS il tempo riportato corrisponde all'attesa CPU. Quando `cuCtxSynchronize ≈ GPU kernel total`, significa che la CPU non fa nulla di utile nel frattempo — è utile per misurare il tempo effettivo di esecuzione ma, se usata dentro un loop, diventa un anti-pattern che serializza CPU e GPU (come dimostrato dalla versione BAD: 2707 ms di sync vs 497 ms di kernel).

- **Shared Memory: quando può aiutare?**
Quando i dati vengono riusati frequentemente da più thread dello stesso blocco, così che il costo di caricare il tile in shared memory venga ammortizzato. Nel caso dello stencil 3×3 (9-point) la shared memory peggiora le prestazioni (overhead > beneficio, riuso basso). Nello stencil 7×7 (49-point) il riuso aumenta ma il beneficio rimane trascurabile su questa GPU, poiché la L2 cache gestisce già bene i pattern di accesso regolari dello stencil.

- **Qual è il bottleneck principale osservato nel report NSYS?**
Il bottleneck principale è il tempo di esecuzione dei kernel GPU (`matmul_naive` e `matmul_tiled`), che occupano ~99.2% del tempo totale. I trasferimenti di memoria e l'overhead CPU sono trascurabili. Il programma è **compute-bound**: è la potenza di calcolo della GPU a limitare le prestazioni. Confermato dal fatto che cuBLAS (CuPy) è ~85x più veloce del kernel Numba tiled, indicando ampio spazio di ottimizzazione nel kernel.

- **Il tempo speso nei trasferimenti è significativo rispetto al calcolo? Perché?**
Dipende dall'esperimento. Per la **vecadd** sì (~67%): il kernel è banale (O(n)) e i trasferimenti pesano quanto il calcolo. Per la **matmul** no (<0.8%): il calcolo è O(n³) e ammortizza i trasferimenti O(n²) su molte iterazioni. In generale, i trasferimenti diventano rilevanti quando il rapporto calcolo/dato (arithmetic intensity) è basso.

- **I kernel CUDA sono lanciati in modo continuo o frammentato nel tempo?**
Continuo in tutti gli esperimenti tranne `sync_bad`. In `sync_good`, `cuCtxSynchronize` (66 ms) << kernel total (1215 ms): la GPU lavora ininterrottamente. In `sync_bad`, `cuCtxSynchronize` (2707 ms) >> kernel total (497 ms): la GPU viene interrotta 20020 volte, creando 2209 ms di overhead da sincronizzazione seriale.

- **In base ai report, quale sarebbe la prima ottimizzazione da tentare?**
La prima ottimizzazione sarebbe sostituire il kernel Numba con **cuBLAS via CuPy**, già ~85x più veloce (da ~730 ms/iter a ~8.5 ms/iter per n=4096). Se si vuole mantenere un kernel custom, si potrebbe aumentare il TILE size (da 16 a 32) per aumentare il riuso in shared memory e l'occupancy degli SM. In alternativa, passare a **mixed-precision FP16** per sfruttare le Tensor Core della GPU, che offrono un throughput teorico molto superiore rispetto a FP32.

E' inoltre richiesto di completare i seguenti esercizi


## Esercizio 1 — Operazione elementwise 1D (Numba vs CuPy)

Implementare l’operazione elementwise:

```math
y[i] = \sin(x[i]) + x[i]^2
```

dove `x` e `y` sono array 1D `float32`.

---

### Parte A — Implementazione Numba

1. Scrivere un kernel CUDA con Numba (`@cuda.jit`) in un file denominato `elem1d_todo_profile.py`.
2. Il kernel deve:
   * supportare array di lunghezza arbitraria;
   * usare `cuda.grid(1)` e un **loop a stride** (come visto nel Lab 1).
3. Scrivere una funzione host che:
   * copi i dati su GPU,
   * lanci il kernel,
   * riporti il risultato su host.
4. Verificare la correttezza confrontando il risultato GPU con:

   ```python
   np.sin(x) + x**2
   ```

   usando `np.allclose`.

---

### Parte B — Implementazione CuPy

5. Implementare la stessa operazione utilizzando **solo CuPy**, senza scrivere kernel CUDA espliciti.
6. L’implementazione deve eseguire l’operazione con funzioni CuPy  (`y_d[...] = cp.sin(x_d) + x_d * x_d` ATTENZIONE: qui `y_d[...]` è corretto e parte dell’implementazione, non è un placeholder)
7. Scrivere una funzione host che:
   * trasferisca i dati su GPU,
   * esegua l’operazione con funzioni CuPy,
   * riporti il risultato su host.
7. Verificare la correttezza confrontando con la versione NumPy.

---

### Parte C — Profilazione e confronto

8. Profilare entrambe le versioni (Numba e CuPy) con **NSYS**.
9. Costruire una tabella che riporti:

| Metodo | Tempo kernel | 
| ------ | ------------ |
| Numba  |   0.0029s    |
| CuPy   |   0.0062s    |
---

## Esercizio 2 — Stencil 2D a 5 punti (Numba vs CuPy)

Dato un input `A` di dimensione **H×W**, calcolare un output `B` di dimensione **(H−2)×(W−2)**:

```math
B[i,j] = A[i,j] + A[i-1,j] + A[i+1,j] + A[i,j-1] + A[i,j+1]
```

---

### Parte A — Implementazione Numba

1. Scrivere un kernel CUDA 2D con Numba (`@cuda.jit`) usando `cuda.grid(2)`.
   Ogni thread deve calcolare un singolo elemento di `B`.
2. Scrivere una funzione host che:
   * allochi input e output su GPU,
   * scelga una configurazione 2D di thread per block,
   * lanci il kernel,
   * riporti il risultato su host.
3. Verificare la correttezza confrontando con una versione NumPy su una matrice **512×512** (`np.allclose`).

---

### Parte B — Implementazione CuPy

4. Implementare lo stesso stencil utilizzando **solo CuPy**, sfruttando slicing e operazioni elementwise.
5. L’output deve avere dimensione **(H−2)×(W−2)**.
6. Verificare la correttezza confrontando con la versione NumPy.

---

### Parte C — Profilazione e confronto

7. Profilare entrambe le versioni (Numba e CuPy) su una matrice grande (es. **4096×4096**).
8. Costruire una tabella che riporti:

| Metodo | Tempo kernel | 
| ------ | ------------ |
| Numba  |    0.0121s   |
| CuPy   |   0.0317s    |


In [20]:
# Profile TODO elem1d
!rm -f results/nsys/elem1d_todo_profile.nsys-rep
!nsys profile --force-overwrite true -t cuda,osrt -o results/nsys/elem1d_todo_profile python scripts/elem1d_todo_profile.py
!python tools/parse_nsys.py results/nsys/elem1d_todo_profile.nsys-rep


Try the 'nsys status --environment' command to learn more.

Try the 'nsys status --environment' command to learn more.

Elem1D max abs diff (numba vs cpu): 1.1920929e-07
Elem1D max abs diff (cupy  vs cpu): 2.3841858e-07
CPU : 0.0189s
Numba (incl one D2H): 0.0029s
CuPy  (incl one D2H): 0.0062s
Generating '/tmp/nsys-report-2b5f.qdstrm'
[1/1] [========================100%] elem1d_todo_profile.nsys-rep
Generated:
	/home/e4user/Documents/s362027/lab2/results/nsys/elem1d_todo_profile.nsys-rep
=== NSYS SUMMARY (student-friendly) ===
CPU wait (cuCtxSynchronize): 2.387 ms
GPU kernel total           : 8.858 ms
GPU memcpy total           : 2.937 ms (HtoD 1.392 ms, DtoH 1.545 ms)

Top 3 kernels by total GPU time:
  1. cupy_add__float32_float32_float32
     total: 2.969 ms | instances: 6
     avg per launch: 0.495 ms
  2. cupy_sin__float32_float32
     total: 1.968 ms | instances: 6
     avg per launch: 0.328 ms
  3. cupy_power__float32_float_float32
     total: 1.968 ms | instances: 6
     avg per

In [21]:
# Profile TODO stencil5
!rm -f results/nsys/stencil5_todo_profile.nsys-rep
!nsys profile --force-overwrite true -t cuda,osrt -o results/nsys/stencil5_todo_profile python scripts/stencil5_todo_profile.py
!python tools/parse_nsys.py results/nsys/stencil5_todo_profile.nsys-rep

Try the 'nsys status --environment' command to learn more.

Try the 'nsys status --environment' command to learn more.

Stencil5 max abs diff (numba vs cpu): 0.0
Stencil5 max abs diff (cupy  vs cpu): 0.0
Stencil5 Numba (4096x4096, incl copies): 0.0121s | out=(4094, 4094)
Stencil5 CuPy  (4096x4096, incl copies): 0.0317s | out=(4094, 4094)
Generating '/tmp/nsys-report-aa7e.qdstrm'
[1/1] [========================100%] stencil5_todo_profile.nsys-rep
Generated:
	/home/e4user/Documents/s362027/lab2/results/nsys/stencil5_todo_profile.nsys-rep
=== NSYS SUMMARY (student-friendly) ===
CPU wait (cuCtxSynchronize): 1.474 ms
GPU kernel total           : 4.729 ms
GPU memcpy total           : 11.101 ms (HtoD 2.530 ms, DtoH 8.571 ms)

Top 2 kernels by total GPU time:
  1. cupy_add__float32_float32_float32
     total: 3.753 ms | instances: 8
     avg per launch: 0.469 ms
  2. stencil5_kernel
     total: 0.976 ms | instances: 2
     avg per launch: 0.488 ms
